# perform deconvolution

In [38]:
import os
import socket
import pandas as pd
import pyarrow
import numpy as np
from scipy.optimize import nnls

print(socket.gethostname())
os.chdir("/home/u24211510018/workspace/Atlas_WGBS/tissue_cell_type_methylation_density")
print(os.getcwd())

cpu10
/home/u24211510018/workspace/Atlas_WGBS/tissue_cell_type_methylation_density


In [39]:
# CpG=5, CV=0.35, Top50 NoDis Hyper and Hypo of Ref Map
reference_map_df = pd.read_parquet("../py_script_data/reference_map_df_CpG5_CV0.35_Top50_NoDis.parquet", engine="pyarrow")
final_marker_annotation = pd.read_parquet("../py_script_data/final_marker_annotation_CpG5_CV0.35_Top50_NoDis.parquet", engine="pyarrow")

In [40]:
final_marker_annotation.shape

(4138, 4)

In [41]:
reference_map_df.shape

(4138, 83)

In [4]:
# 按照 'tissue' 列分组，统计每个组合的行数
tissue_counts = final_marker_annotation.groupby('tissue').size()

# 显示结果
print(tissue_counts)

tissue
Adipocytes                           41
Adipocytes,Endometrium-Epithelial     1
Adipocytes,Pancreas-Endothel          1
Adipocytes,Prostate-Smooth-Muscle     2
Aorta-Endothel                       50
                                     ..
Thyroid-Epithelial                   50
Tongue-Epithelial                    50
Tongue_base-Epithelial               50
Tonsil-Palatine-Epithelial           50
Tonsil-Pharyngeal-Epithelial         50
Length: 89, dtype: int64


In [5]:
final_marker_annotation["cpg_count"].mean()

np.float64(19.153455775737072)

In [6]:
reference_map_df.head()

,Adipocytes,Aorta-Endothel,Aorta-Smooth-Muscle,Bladder-Epithelial,Bladder-Smooth-Muscle,Blood-B,Blood-B-Mem,Blood-Granulocytes,Blood-Monocytes,Blood-NK,...,Prostate-Smooth-Muscle,Saphenous-Vein-Endothel,Skeletal-Muscle,Small-int-Endocrine,Small-int-Epithelial,Thyroid-Epithelial,Tongue-Epithelial,Tongue_base-Epithelial,Tonsil-Palatine-Epithelial,Tonsil-Pharyngeal-Epithelial
chr18_79358000_79358500,67.405333,20.8330,27.083,29.0000,8.333,22.483333,15.2750,15.278000,26.666667,18.027667,...,11.117,12.039000,17.2250,11.900,24.98750,23.922000,19.283333,27.783,9.227667,34.3750
chr5_181053500_181054000,21.302000,5.2405,3.851,5.5152,4.864,6.222333,4.3410,4.790667,4.386000,5.502333,...,4.701,4.660000,2.9145,3.539,3.40050,3.505667,4.183333,3.838,3.517667,2.3180
chr8_127391500_127392000,16.136667,46.1000,31.464,46.8748,24.076,80.840000,81.1000,43.970000,52.857333,82.380000,...,27.952,37.666000,39.3490,94.880,92.00000,88.953333,35.123333,34.226,36.710667,39.3190
chr22_38317000_38317500,11.347667,2.3370,4.028,3.5728,2.410,4.122667,2.1875,3.014000,2.812667,3.956000,...,1.850,6.769333,3.0475,1.701,2.89375,1.205000,3.584000,3.333,2.414000,3.4275
chr1_153774500_153775000,13.373667,26.5775,24.642,41.6622,22.517,76.211000,79.6250,44.950667,56.403333,53.824000,...,23.938,25.038333,50.7345,52.332,51.48400,59.358667,28.226667,26.417,29.098000,36.3525


In [55]:
# using public data to test the rep map.
sample_files = {
    "ENCODE_Pancreas1": "/home/u24211510018/workspace/Atlas_WGBS/Pancreas/ENCFF080TDH_CpG_dyad.methylation_density.bed",
    "ENCODE_Pancreas2": "/home/u24211510018/workspace/Atlas_WGBS/Pancreas/ENCFF689HVV_CpG_dyad.methylation_density.bed",
    "breast_tumor": "/home/u24211510018/workspace/Atlas_WGBS/asTair_result/breast_tumor_human_CpG_dyad.methylation_density.bed",
}

In [56]:
def read_methylation_bed(file_path):
    df = pd.read_csv(
        file_path,
        sep="\t",
        header=None,
        names=["chr", "start", "end", "value"]
    )
    
    df["region_id"] = (
        df["chr"].astype(str) + "_" +
        df["start"].astype(str) + "_" +
        df["end"].astype(str)
    )
    
    s = pd.Series(df["value"].values, index=df["region_id"].values, name=os.path.basename(file_path))
    return s

In [57]:
sample_series_dict = {}

for sample_name, file_path in sample_files.items():
    sample_series_dict[sample_name] = read_methylation_bed(file_path)

for k, v in sample_series_dict.items():
    print(k, v.shape)
    print(v.head())

ENCODE_Pancreas1 (5008612,)
chr1_10000_10500    92.367
chr1_10500_11000    57.346
chr1_13000_13500    50.780
chr1_14500_15000    49.581
chr1_15000_15500    22.407
Name: ENCFF080TDH_CpG_dyad.methylation_density.bed, dtype: float64
ENCODE_Pancreas2 (5004946,)
chr1_10000_10500    67.217
chr1_10500_11000    33.683
chr1_13000_13500    57.500
chr1_14000_14500     0.000
chr1_14500_15000    53.125
Name: ENCFF689HVV_CpG_dyad.methylation_density.bed, dtype: float64
breast_tumor (4943206,)
chr1_10000_10500    95.233
chr1_10500_11000    17.073
chr1_12500_13000    14.286
chr1_13000_13500    62.780
chr1_13500_14000    33.510
Name: breast_tumor_human_CpG_dyad.methylation_density.bed, dtype: float64


In [58]:
def nnls_deconvolution(sample_series, reference_map_df, min_common_blocks=100):
    # 找共同 block
    common_blocks = reference_map_df.index.intersection(sample_series.index)
    
    if len(common_blocks) < min_common_blocks:
        raise ValueError(f"共同 block 太少: {len(common_blocks)}，小于阈值 {min_common_blocks}")
    
    # 参考矩阵 X：行为 block，列为 tissue
    X = reference_map_df.loc[common_blocks].values.astype(float)
    
    # 待分解样本 y：行为 block
    y = sample_series.loc[common_blocks].values.astype(float)
    
    # NNLS
    coef, residual = nnls(X, y)
    
    # 归一化为比例
    coef_sum = coef.sum()
    if coef_sum > 0:
        coef_prop = coef / coef_sum
    else:
        coef_prop = coef
    
    result_df = pd.DataFrame({
        "tissue": reference_map_df.columns,
        "coefficient": coef,
        "proportion": coef_prop
    }).sort_values("proportion", ascending=False).reset_index(drop=True)
    
    return result_df, residual, len(common_blocks)

In [59]:
pd.set_option('display.max_rows', None)
deconv_results = {}

for sample_name, sample_series in sample_series_dict.items():
    result_df, residual, n_blocks = nnls_deconvolution(
        sample_series=sample_series,
        reference_map_df=reference_map_df,
        min_common_blocks=100
    )
    
    deconv_results[sample_name] = {
        "result_df": result_df,
        "residual": residual,
        "n_common_blocks": n_blocks
    }
    
    print("=" * 60)
    print(sample_name)
    print("共同 block 数:", n_blocks)
    print("residual:", residual)
    print(result_df.head(83))

ENCODE_Pancreas1
共同 block 数: 4107
residual: 437.7591038289946
                                 tissue  coefficient  proportion
0                       Pancreas-Acinar     0.653705    0.670674
1                         Pancreas-Duct     0.206556    0.211918
2                     Pancreas-Endothel     0.032923    0.033778
3                        Megakaryocytes     0.030332    0.031119
4                   Aorta-Smooth-Muscle     0.024442    0.025077
5         Coronary-Artery-Smooth-Muscle     0.015831    0.016242
6          Kidney-Glomerular-Epithelial     0.003183    0.003266
7                         Pancreas-Beta     0.002719    0.002789
8                     Colon-Macrophages     0.002588    0.002655
9                     Cerebellum-Neuron     0.001403    0.001439
10              Pancreas-Islet-Endothel     0.000549    0.000563
11               Prostate-Smooth-Muscle     0.000396    0.000407
12               Endometrium-Epithelial     0.000072    0.000073
13                          